In [1]:
#| code-fold: fold
import sys
from pathlib import Path

module_dir = Path(".").resolve()
sys.path.append(str(module_dir))

import osmnx as ox
from network_explorer import BikeNetworkExplorer

On a trip to Copenhagen, I biked in a city for the first time in a decade. It was delightful and liberating, and it made sense that the majority of Copenhagen residents primarily commuted by bike. Returning home to Chicago, I bought a bike and quickly felt the difference that bike infrastructure and culture makes.

In Copenhagen, bike safety and bike theft were basically solved problems. Extensive infrastructure made routes everywhere safe to bike and from the number of unlocked bikes, it was clear that bike theft wasn't an issue. Back in Chicago, I was sharing lanes with cars going 40 mph and despite extensive locking and uglying up my bike, I still never know if my bike will still be there. Over time I've learned safer, more comfortable routes, and Chicago keeps building out new infrastructure, but the gap between those two experiences stuck with me. So I built a tool to help find the safer routes and find secure parking areas.

I wanted to gain a richer understanding of how routing algorithms work, and I wanted to find routes that optimize for cyclist safety instead of speed or distance. The result is a platform that scores every road segment in the Chicago area for danger — using crash data, bike infrastructure, speed limits, and road type — and routes you through the safest path, not the shortest one. It also maps bike thefts and parking infrastructure so you can figure out where it's safe to lock up.

You can try it at [bike-map.dev.missinglastmile.net](https://bike-map.dev.missinglastmile.net/).

| ![elt dags in airflow](imgs/bike_map__palmer_park_to_wrigley.png) | ![querying through superset](imgs/google_maps__palmer_park_to_wrigley.png) |
|:--:|:--:|
| Bike-Maps Safety-Optimized Route | Google Maps Routes |

## How routing algorithms work

You already have an intuition for this, even if you've never thought about it in these terms.

Think about a route you ride or drive regularly. At each intersection, you know which way to go — not because you've evaluated every possible path through the city, but because you can tell which options move you toward your destination and which ones don't. You don't even consider turning away from where you're going unless you have a good reason, like avoiding a busy street or to get over a bridge.

That's basically how a routing algorithm works. It starts at Point A and looks at the options. At each intersection, it asks: which of these roads gets me closer to Point B at the lowest cost? And it holds off on exploring options that move in the wrong direction (until and unless it discovers the more direct options dead-end).

In computer science, that "explore the most likely options first" instinct has a name: a **heuristic**. The specific routing algorithm I use, the A* (pronounced "A star") algorithm, works exactly this way. It combines the actual cost of the path so far with an estimate of the remaining distance, and always explores the most direct option next.

The vocabulary maps directly onto the physical world:

- A **graph** (or **network**) is just a street grid — a set of locations connected by roads.
- A **node** is an intersection (or any point where roads meet or end).
- An **edge** is a road segment between two intersections.
- A **cost** is whatever you'd "pay" by traveling down that segment. It could be distance, time, effort, risk — anything.

Here's what a few blocks of Chicago's actual bike network look like as a graph:

In [2]:
#| code-fold: show
#| code-summary: Examining the raw bike network

west = -87.6620; south = 41.8959; east = -87.6423; north = 41.9113
goose_island_graph = ox.graph_from_bbox(
    bbox=(west, south, east, north), network_type="bike", retain_all=True
)
goose_island_explorer = BikeNetworkExplorer(goose_island_graph)
goose_island_explorer.show(zoom_start=15)

Every line on that map is a [bicycle-relevant](https://wiki.openstreetmap.org/wiki/Bicycle) edge in the OpenStreetMaps (OSM) data, and every dot where lines intersect is a node. Different colors indicate different disconnected networks (aka Components in the graph; note that the largest component has ~2200 edges and the second largest component has less than 20; that largest component is highly connected), and clicking on any node or edge will provide more detail about that element. A routing algorithm can find a path through this network from an origin node (Point A) to a destination node (Point B) that minimizes some total cost.

The key insight is that **cost can be anything**. When Google Maps routes you, the cost on each edge is usually travel time. When my app routes you, the cost is a safety-weighted score, a number that estimates the risk a cyclist incurs on a trip over each road segment, where that risk is based on crash history, bike infrastructure, speed limits, road type, and traffic control devices. The algorithm doesn't know what the cost number represents, it just finds the path where the numbers add up to the smallest total. But if those cost numbers accurately reflect the risk, paths that minimize that number are safer paths.

So, how do use messy real-world data to accurately price out the risk of traveling each edge?

## What does "safe" mean for a road segment?

This is the interesting question at the heart of the project. Google Maps and other routing tools optimize for time or distance. Some have basic bike modes that prefer bike lanes when they exist. But none of them let you define what "safe" means to you, or plug in local crash data to inform the route.

I wanted to combine several factors:

- **Crash history**: where have cyclists actually been hit? More recent crashes weigh more heavily (largely as I don't know when infrastructure was created and want to downweight old crashes in case new infrastructure has been added).
- **Bike infrastructure**: is there a protected lane, a painted bike lane, a sharrow, or nothing at all? How close can cars get to you?
- **Speed limits**: a 25 mph residential street is a very different environment from a 40 mph arterial.
- **Road type**: a dedicated bike path is inherently different from a four-lane road, even before you look at infrastructure or crash data.
- **Traffic control devices**: Intersections are the most dangerous place to ride and that's somewhat mitigated by a traffic control device (e.g. a stop sigh, traffic light, etc).

Each road segment gets a cost that estimates the risk of riding down that segment, and the routing algorithm finds the path with the lowest total score — the safest route, not the shortest one.

![avoiding a high crash area](imgs/avoiding_high_crash_areas_when_infra_allows.png)

## What I actually built

The platform has a few pieces:

- **Data collection**: I pull road network data from OpenStreetMap, crash data and bike infrastructure data from the City of Chicago's open data portal, and bike parking fixture locations.
- **Transformation**: I use `dbt` to prepare and assemble that data into a form that supports application needs. [Here](https://github.com/MattTriano/loci_platform/blob/2f9358c154bda525f6b508900e71b2e08ad79dc0/platform/dbt/models/marts/cycling/bike_safety_weighted_edges.sql) is the model that calculates the safety-weighted cost for each edge (or route segment) in the collected bike network.
- **Graph export**: the road network with safety-weighted costs gets exported as a graph file and uploaded to S3.
- **Routing API**: an AWS Lambda function loads the graph into memory and runs A* pathfinding when someone requests a route. If you exclude the $17 per year cost of the custom domain name, I pay AWS $0.00 per month to operate this web app.
- **Frontend**: a MapLibre-based web map that lets you explore crash data, theft data, and bike parking, and request safety-weighted routes between any two points.


| ![building out the data and deploying it](imgs/crash_data_collection_dag.png) | ![building out the data and deploying it](imgs/app_deployment_dag.png) |
|:--:|:--:|
| Example of one of the data collection DAGs | The DAG that transforms collected data, exports it, and deployes it along with the application to AWS |



## The theft and parking problem

The routing gets the most attention, but the data layers might be more useful day-to-day. The app shows where bike thefts are concentrated and where the city's bike parking fixtures are — racks, corrals, etc. Before I ride somewhere I haven't been, I check whether there's a reasonable place to lock up. That's the Copenhagen problem I still haven't fully solved in Chicago, but at least now I can make informed decisions about it.

| ![zoomed out view](imgs/bike_parking_aggregated_counts.png) |
|:--:|
| Zoomed out, you'll see aggregated counts of crashes, thefts, and bike parking locations |

| ![zoomed in view](imgs/bike_parking_zoomed_in.png) |
|:--:|
| Zoomed in, you'll see point locations that you can select for more detail |

Notice how the route goes three blocks west to travel north on a residential street with no crashes instead of the shorter route straight north on Halstead (over a four block span with ~20 CPD-recorded crashes over the past three years). Routes like that tell me my cost estimation model is having the desired effect, namely that it's setting a high enough cost for risky route segments that the algorithm finds that it's less expensive (in terms of risk) to take a slightly longer, much more pleasant route.

## Try it out

The app is live at [bike-map.dev.missinglastmile.net](https://bike-map.dev.missinglastmile.net/). Pick two points in Chicago and see what route it suggests. Toggle the crash and theft layers to see how risk is distributed across the city.

I'd genuinely love feedback, especially from people who ride in Chicago. Do the routes match your experience? Are there roads the app suggests that you'd avoid, or roads it avoids that you think are fine? That kind of local knowledge is exactly what helps me tune the model.

If you're interested in the technical details — the dbt models, the Lambda deployment, or how I penalize left turns — I'll be writing more about specific pieces of the platform in future posts.
